In [ ]:
!pip install -q transformers peft bitsandbytes accelerate pandas tqdm

In [ ]:
import os
import re
import json
import torch
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
# change the path to your own drive path or local path
ADAPTER_PATH = "/content/drive/MyDrive/Deep Learning Mid-Term/Model_Output"
DATA_DIR = "/content/drive/MyDrive/Deep Learning Mid-Term/dl-spring-2026-svg-generation"

test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Loaded successfully")

In [ ]:
SYSTEM_PROMPT = "Generate one valid SVG only."

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon", "defs", "use", "symbol", "clipPath", "mask", "linearGradient", "radialGradient", "stop", "text", "tspan", "title", "desc", "style", "pattern", "marker", "filter"
}

SVG_MAX_LEN = 16000
SVG_MAX_PATHS = 256
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def strip_ns(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

def extract_svg(text):
    m = SVG_REGEX.search(str(text))
    return m.group(0).strip() if m else ""

def validate_svg(svg_text):
    if not svg_text or len(svg_text) > SVG_MAX_LEN:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if strip_ns(root.tag) != "svg":
        return False
    path_count = 0
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
    return path_count <= SVG_MAX_PATHS

def sanitize_svg(svg_text):
    svg_text = extract_svg(svg_text) if "<svg" in str(svg_text).lower() else str(svg_text).strip()
    if not svg_text:
        return ""
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return ""
    if strip_ns(root.tag) != "svg":
        return ""
    root.set("xmlns", "http://www.w3.org/2000/svg")
    root.set("width", "256")
    root.set("height", "256")
    root.set("viewBox", "0 0 256 256")
    cleaned = ET.tostring(root, encoding="unicode")
    return cleaned if validate_svg(cleaned) else ""

def fallback_svg(prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="48" fill="#888"/></svg>'
    )

In [ ]:
def generate_svg_batch(prompts, max_new_tokens=180):
    messages_list = [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(prompt)},
        ]
        for prompt in prompts
    ]

    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_list
    ]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=896,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.5,
            top_p=0.95,
            repetition_penalty=1.03,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    svgs = []
    for text, prompt in zip(decoded, prompts):
        svg = sanitize_svg(text)
        if not svg:
            svg = fallback_svg(prompt)
        svgs.append(svg)
    return svgs

In [ ]:
sample_prompts = test_df["prompt"].head(50).tolist()
sample_svgs = generate_svg_batch(sample_prompts, max_new_tokens=180)

print("valid rate:", sum(validate_svg(x) for x in sample_svgs) / len(sample_svgs))
print("unique count:", len(set(sample_svgs)))
print("avg len:", sum(len(x) for x in sample_svgs) / len(sample_svgs))
print(sample_svgs[0][:500])

In [ ]:
rows = []
batch_size = 8

for i in tqdm(range(0, len(test_df), batch_size)):
    batch_prompts = test_df["prompt"].iloc[i:i+batch_size].tolist()
    batch_ids = test_df["id"].iloc[i:i+batch_size].tolist()

    batch_svgs = generate_svg_batch(batch_prompts, max_new_tokens=180)

    for id_, svg, prompt in zip(batch_ids, batch_svgs, batch_prompts):
        if not validate_svg(svg):
            svg = fallback_svg(prompt)
        rows.append({"id": id_, "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv("/content/submission.csv", index=False) # Save the submission file to your desired location

print(sub_df.shape)
sub_df.head()